In [1]:
"""Thí nghiệm 1 - Bước 03: Feature Engineering chỉ từ OHLCV.

Cách chạy: python 03_feature_engineering_EXP1.py
Thư viện: pandas, numpy (Python 3.10+).

- Đọc đầu ra Bước 02 (70/15/15), KHÔNG chia lại.
- Tính bốn đặc trưng thô từ OHLCV theo mã và ngày.
- So sánh từng phiên với đúng W giá trị đặc trưng trước đó của cùng mã.
- Học mức sàn mẫu số CHỈ từ Train; áp dụng giữ nguyên cho Val/Test.
- Không huấn luyện IF/AE/LOF, không tạo proxy và không xem điểm Test.
"""

from __future__ import annotations

import hashlib
import json
from pathlib import Path

import numpy as np
import pandas as pd
from numpy.lib.stride_tricks import sliding_window_view

# ---------- 1. CẤU HÌNH ----------
# Đổi thành đường dẫn thư mục đầu ra Bước 02 trên máy của bạn.
INPUT_DIR = Path(r"outputs/02_time_split_70_15_15")
OUTPUT_DIR = Path(r"outputs/03_feature_engineering")
W = 60  # Cấu hình KHỞI ĐẦU, sẽ kiểm tra độ nhạy trên Valid chọn cấu hình
MAD_NORMAL_FACTOR = 1.4826
FLOOR_QUANTILE = 0.10  # Phân vị dương của thang MAD trên TRAIN, chưa phải tối ưu

PRICE_COLS = ["open", "high", "low", "close"]
SOURCE_COLS = ["ticker", "date", *PRICE_COLS, "volume"]
KEY_COLS = ["ticker", "date"]
RAW_MAP = {
    "gap": "raw_gap",             # ln(Open_t / Close_{t-1})
    "intraday": "raw_intraday",   # ln(Close_t / Open_t)
    "range": "raw_range",         # ln(High_t / Low_t)
    "volume": "raw_log_volume",  # ln(1 + Volume_t)
}
FEATURE_COLS = [f"z_{k}" for k in RAW_MAP]




In [2]:
def sha256(path: Path) -> str:
    h = hashlib.sha256()
    with path.open("rb") as f:
        for part in iter(lambda: f.read(1024 * 1024), b""):
            h.update(part)
    return h.hexdigest()


def read_and_check_inputs() -> pd.DataFrame:
    source_file = INPUT_DIR / r"02_ohlcv_with_split.csv"
    manifest_file = INPUT_DIR / r"02_split_manifest.csv"
    config_file = INPUT_DIR / r"02_split_config.json"
    for p in (source_file, manifest_file, config_file):
        if not p.is_file():
            raise FileNotFoundError(f"Thiếu tệp Bước 02: {p.resolve()}")

    with config_file.open(encoding="utf-8") as f:
        config = json.load(f)
    expected_ratios = {"train": 0.7, "val": 0.15, "test": 0.15}
    if config.get("target_ratios") != expected_ratios:
        raise ValueError("Cấu hình Bước 02 không phải 70/15/15")

    d = pd.read_csv(source_file, encoding="utf-8-sig")
    required = SOURCE_COLS + ["split", "val_role"]
    if set(d.columns) != set(required):
        raise ValueError(f"Cột Bước 02 không khớp: {d.columns.tolist()}")
    d = d[required].copy()
    d["date"] = pd.to_datetime(d["date"], format="%Y-%m-%d", errors="raise")
    d["ticker"] = d["ticker"].astype("string").str.strip()
    for col in PRICE_COLS + ["volume"]:
        d[col] = pd.to_numeric(d[col], errors="raise")

    if d.isna().any().any() or d.duplicated(KEY_COLS).any():
        raise ValueError("Thiếu giá trị hoặc trùng khóa mã-ngày")
    if not np.isfinite(d[PRICE_COLS + ["volume"]].to_numpy(dtype=float)).all():
        raise ValueError("Có OHLCV không hữu hạn")
    if not (d[PRICE_COLS] > 0).all().all() or not (d["volume"] >= 0).all():
        raise ValueError("OHLC phải dương, Volume không âm")
    if not (d["high"] >= d[["open", "low", "close"]].max(axis=1)).all():
        raise ValueError("High không hợp lệ")
    if not (d["low"] <= d[["open", "high", "close"]].min(axis=1)).all():
        raise ValueError("Low không hợp lệ")

    manifest = pd.read_csv(manifest_file, encoding="utf-8-sig")
    manifest["date"] = pd.to_datetime(manifest["date"], format="%Y-%m-%d")
    cols = KEY_COLS + ["split", "val_role"]
    pd.testing.assert_frame_equal(
        d[cols].sort_values(KEY_COLS).reset_index(drop=True),
        manifest[cols].sort_values(KEY_COLS).reset_index(drop=True),
        check_dtype=False,
    )

    boundaries = config["boundaries"]
    train_end = pd.Timestamp(boundaries["train_end"])
    val_end = pd.Timestamp(boundaries["val_end"])
    select_end = pd.Timestamp(boundaries["val_select_end"])
    split_expected = np.select(
        [d["date"] <= train_end, d["date"] <= val_end],
        ["train", "val"], default="test",
    )
    if not (d["split"].to_numpy() == split_expected).all():
        raise ValueError("Có dòng mang nhãn split không khớp mốc ngày")
    role_expected = np.select(
        [
            (d["split"] == "val") & (d["date"] <= select_end),
            (d["split"] == "val") & (d["date"] > select_end),
        ],
        ["select", "calibrate"], default="not_applicable",
    )
    if not (d["val_role"].to_numpy() == role_expected).all():
        raise ValueError("Có dòng mang nhãn val_role không khớp mốc ngày")
    if d["ticker"].nunique() != 13:
        raise ValueError("Số mã không phải 13 như phạm vi hiện tại")
    if len(d) != 15977:
        raise ValueError("Số dòng khác bản Bước 02 đã kiểm tra")
    return d.sort_values(["ticker", "date"]).reset_index(drop=True)




In [3]:
def make_raw_features(d: pd.DataFrame) -> pd.DataFrame:
    """Không dùng giá tham chiếu hay bất kỳ cột dữ liệu ngoài 7 trường OHLCV."""
    d = d.copy()
    d["prev_valid_close"] = d.groupby("ticker", sort=False)["close"].shift(1)
    d[RAW_MAP["gap"]] = np.log(d["open"] / d["prev_valid_close"])
    d[RAW_MAP["intraday"]] = np.log(d["close"] / d["open"])
    d[RAW_MAP["range"]] = np.log(d["high"] / d["low"])
    d[RAW_MAP["volume"]] = np.log1p(d["volume"])
    # Giá đóng cửa trước = phiên hợp lệ gần nhất trong dữ liệu clean.
    # 13 phiên IMP không khớp lệnh đã bị loại ở Bước 01 và KHÔNG được đưa lại.
    return d


def strictly_past_median_mad(values: np.ndarray, window: int):
    """Tại t: median và MAD của values[t-window:t], không chứa t.

    MAD_t = median(|x_i - median(cửa sổ)|), không phải rolling median
    của các độ lệch quanh những median theo thời gian khác nhau.
    """
    x = np.asarray(values, dtype=float)
    n = len(x)
    med = np.full(n, np.nan, dtype=float)
    mad = np.full(n, np.nan, dtype=float)
    if n <= window:
        return med, mad
    windows = sliding_window_view(x, window_shape=window)[:-1]
    enough = np.isfinite(windows).all(axis=1)
    computed_med = np.full(len(windows), np.nan, dtype=float)
    computed_mad = np.full(len(windows), np.nan, dtype=float)
    if enough.any():
        w_ok = windows[enough]
        m = np.median(w_ok, axis=1)
        computed_med[enough] = m
        computed_mad[enough] = np.median(np.abs(w_ok - m[:, None]), axis=1)
    med[window:] = computed_med
    mad[window:] = computed_mad
    return med, mad


def make_rolling_references(d: pd.DataFrame, window: int) -> pd.DataFrame:
    d = d.copy()
    for name in RAW_MAP:
        d[f"hist_median_{name}"] = np.nan
        d[f"hist_mad_{name}"] = np.nan
    for _, group in d.groupby("ticker", sort=False):
        indices = group.index
        for name, raw_col in RAW_MAP.items():
            med, mad = strictly_past_median_mad(group[raw_col].to_numpy(), window)
            d.loc[indices, f"hist_median_{name}"] = med
            d.loc[indices, f"hist_mad_{name}"] = mad
    return d




In [4]:
def fit_train_floors_and_standardize(d: pd.DataFrame):
    d = d.copy()
    raw_cols = list(RAW_MAP.values())
    med_cols = [f"hist_median_{k}" for k in RAW_MAP]
    mad_cols = [f"hist_mad_{k}" for k in RAW_MAP]
    raw_ok = np.isfinite(d[raw_cols].to_numpy(dtype=float)).all(axis=1)
    hist_ok = np.isfinite(d[med_cols + mad_cols].to_numpy(dtype=float)).all(axis=1)
    candidate = raw_ok & hist_ok
    train_candidate = candidate & d["split"].eq("train").to_numpy()
    if train_candidate.sum() == 0:
        raise ValueError("Train không có mẫu có đủ bốn đặc trưng và nền lịch sử")

    floors = {}
    for name, raw_col in RAW_MAP.items():
        scale = MAD_NORMAL_FACTOR * d[f"hist_mad_{name}"].to_numpy(dtype=float)
        positive_train = scale[train_candidate & np.isfinite(scale) & (scale > 0)]
        if len(positive_train) == 0:
            raise ValueError(f"Train không có MAD dương cho thành phần {name}")
        floor = float(np.quantile(positive_train, FLOOR_QUANTILE))
        if not np.isfinite(floor) or floor <= 0:
            raise ValueError(f"Mức sàn không hợp lệ cho {name}: {floor}")
        floors[name] = floor

        denominator = np.maximum(scale, floor)
        result = (d[raw_col].to_numpy() - d[f"hist_median_{name}"].to_numpy()) / denominator
        result[~candidate] = np.nan  # Chỉ xuất vector hoàn chỉnh khi đủ cả 4 thành phần
        d[f"z_{name}"] = result
        d[f"mad_zero_{name}"] = (d[f"hist_mad_{name}"] == 0) & candidate
        d[f"floor_applied_{name}"] = (scale < floor) & candidate

    d["feature_status"] = np.select(
        [~raw_ok, raw_ok & ~hist_ok],
        ["invalid_raw_feature", "insufficient_valid_history"],
        default="ok",
    )
    d["is_model_ready"] = d["feature_status"].eq("ok")
    ready = d.loc[d["is_model_ready"], FEATURE_COLS].to_numpy(dtype=float)
    if not np.isfinite(ready).all():
        raise ValueError("Có giá trị đặc trưng chuẩn hóa không hữu hạn")
    if not d.loc[~d["is_model_ready"], FEATURE_COLS].isna().all().all():
        raise ValueError("Phiên chưa đủ điều kiện vẫn có đặc trưng chuẩn hóa")
    return d, floors


def independent_qa(d: pd.DataFrame, floors: dict) -> None:
    """Kiểm tra toán học và không dùng tương lai. Không chọn mô hình theo Test."""
    for ticker, group in d.groupby("ticker", sort=False):
        group = group.reset_index(drop=True)
        assert group.loc[0, "feature_status"] == "invalid_raw_feature"
        assert group.loc[:W, "is_model_ready"].sum() == 0
        if len(group) > W + 1:
            assert bool(group.loc[W + 1, "is_model_ready"]), (ticker, W)
            j = W + 1
            for name, raw_col in RAW_MAP.items():
                past = group.loc[j-W:j-1, raw_col].to_numpy(dtype=float)
                m = float(np.median(past))
                mad = float(np.median(np.abs(past - m)))
                np.testing.assert_allclose(group.loc[j, f"hist_median_{name}"], m, rtol=0, atol=1e-12)
                np.testing.assert_allclose(group.loc[j, f"hist_mad_{name}"], mad, rtol=0, atol=1e-12)
                expected_z = (group.loc[j, raw_col] - m) / max(MAD_NORMAL_FACTOR * mad, floors[name])
                np.testing.assert_allclose(group.loc[j, f"z_{name}"], expected_z, rtol=0, atol=1e-10)

    # Kiểm tra đồng nhất thức gap + intraday = log(C / C_prev).
    usable = d["prev_valid_close"].notna()
    np.testing.assert_allclose(
        (d.loc[usable, "raw_gap"] + d.loc[usable, "raw_intraday"]).to_numpy(),
        np.log((d.loc[usable, "close"] / d.loc[usable, "prev_valid_close"]).to_numpy()),
        atol=1e-12, rtol=0,
    )
    # Phiên High = Low hợp lệ => log(H / L) = 0, không bị loại.
    flat = d["high"].eq(d["low"])
    assert np.allclose(d.loc[flat, "raw_range"], 0.0, atol=1e-15)

    # Kiểm tra tính nhân quả: tính lại tham chiếu trên tiền tố Train,
    # các giá trị trước ranh giới phải giống hệt khi có thêm dữ liệu tương lai.
    train_end = d.loc[d["split"].eq("train"), "date"].max()
    prefix = d.loc[d["date"] <= train_end, SOURCE_COLS + ["split", "val_role"]].copy()
    prefix = make_rolling_references(make_raw_features(prefix.reset_index(drop=True)), W)
    old = d.loc[d["date"] <= train_end].sort_values(KEY_COLS).reset_index(drop=True)
    new = prefix.sort_values(KEY_COLS).reset_index(drop=True)
    for col in list(RAW_MAP.values()) + [f"hist_{stat}_{name}" for name in RAW_MAP for stat in ("median", "mad")]:
        np.testing.assert_allclose(old[col], new[col], rtol=0, atol=0, equal_nan=True)




In [5]:
def main() -> None:
    if W < 2 or not 0 < FLOOR_QUANTILE < 1:
        raise ValueError("W phải >=2 và FLOOR_QUANTILE nằm trong (0,1)")
    source_file = INPUT_DIR / "02_ohlcv_with_split.csv"
    manifest_file = INPUT_DIR / "02_split_manifest.csv"
    d = read_and_check_inputs()
    d = make_raw_features(d)
    d = make_rolling_references(d, W)
    d, floors = fit_train_floors_and_standardize(d)
    independent_qa(d, floors)

    summary = (
        d.groupby(["split", "val_role", "feature_status"], dropna=False)
        .size().rename("sessions").reset_index()
    )
    by_ticker = (
        d.groupby(["ticker", "split"], dropna=False)
        .agg(rows=("date", "size"), model_ready=("is_model_ready", "sum"))
        .reset_index()
    )
    train = d.loc[d["split"].eq("train")].copy()
    train_diagnostics = {
        name: {
            "floor": floors[name],
            "mad_zero_train_ready": int(train.loc[train["is_model_ready"], f"mad_zero_{name}"].sum()),
            "floor_applied_train_ready": int(train.loc[train["is_model_ready"], f"floor_applied_{name}"].sum()),
        }
        for name in RAW_MAP
    }
    info = {
        "input_02_ohlcv_sha256": sha256(source_file),
        "input_02_manifest_sha256": sha256(manifest_file),
        "source_columns": SOURCE_COLS,
        "split_rule": "reuse Step 02 labels; no split refit",
        "window_prior_valid_feature_values_per_ticker": W,
        "mad_definition": "median(abs(past_window - median(past_window)))",
        "mad_normal_factor": MAD_NORMAL_FACTOR,
        "train_floor_quantile_positive_mad_scales": FLOOR_QUANTILE,
        "train_floors": floors,
        "train_diagnostics": train_diagnostics,
        "features_for_model": FEATURE_COLS,
        "normalization": "z=(raw - past_median)/max(1.4826*past_MAD, train_floor)",
        "first_gap_rule": "prior valid trading session close within cleaned per-ticker history",
        "test_policy": "Causal Test features computed; no Test analysis used to choose W, floor or features",
    }
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    d = d.sort_values(["date", "ticker"]).reset_index(drop=True)
    d["date"] = d["date"].dt.strftime("%Y-%m-%d")
    d.to_csv(OUTPUT_DIR / "03_features_all.csv", index=False, encoding="utf-8-sig")
    # Chỉ xuất các mẫu Train/Validation làm đầu vào phát triển mô hình.
    # Test ở bảng audit đầy đủ nhưng KHÔNG được đưa vào file huấn luyện.
    development = d["is_model_ready"] & d["split"].isin(["train", "val"])
    d.loc[development, KEY_COLS + ["split", "val_role"] + FEATURE_COLS].to_csv(
        OUTPUT_DIR / "03_model_ready_development.csv", index=False, encoding="utf-8-sig"
    )
    summary.to_csv(OUTPUT_DIR / "03_feature_status_summary.csv", index=False, encoding="utf-8-sig")
    by_ticker.to_csv(OUTPUT_DIR / "03_feature_coverage_by_ticker.csv", index=False, encoding="utf-8-sig")
    with (OUTPUT_DIR / "03_feature_config.json").open("w", encoding="utf-8") as f:
        json.dump(info, f, indent=2, ensure_ascii=False)

    # Không in phân bố đặc trưng Test để tránh lựa chọn phương án theo Test.
    print(f"Đã đọc {len(d):,} phiên của {d['ticker'].nunique()} mã.")
    print(f"W={W}. Số mẫu đủ đặc trưng theo tập:")
    print(d.groupby("split")["is_model_ready"].agg(["size", "sum"]).to_string())
    print("\nTrạng thái chưa đủ đặc trưng:")
    print(d["feature_status"].value_counts().to_string())
    print("\nMức sàn đã FIT từ Train:")
    for name, values in train_diagnostics.items():
        print(f"{name}: floor={values['floor']:.10g}, MAD=0 (Train ready)={values['mad_zero_train_ready']}, "
              f"dùng sàn (Train ready)={values['floor_applied_train_ready']}")
    print("\nHoàn tất Bước 03 về tạo đặc trưng. Chưa huấn luyện/chọn ngưỡng/đánh giá Test.")
    print("Đầu ra:", OUTPUT_DIR.resolve())




In [6]:
if __name__ == "__main__":
    main()


Đã đọc 15,977 phiên của 13 mã.
W=60. Số mẫu đủ đặc trưng theo tập:
        size    sum
split              
test    2405   2405
train  11180  10387
val     2392   2392

Trạng thái chưa đủ đặc trưng:
feature_status
ok                            15184
insufficient_valid_history      780
invalid_raw_feature              13

Mức sàn đã FIT từ Train:
gap: floor=0.002092594495, MAD=0 (Train ready)=30, dùng sàn (Train ready)=1062
intraday: floor=0.00858903476, MAD=0 (Train ready)=0, dùng sàn (Train ready)=1037
range: floor=0.006032505753, MAD=0 (Train ready)=0, dùng sàn (Train ready)=1038
volume: floor=0.3611682033, MAD=0 (Train ready)=0, dùng sàn (Train ready)=1038

Hoàn tất Bước 03 về tạo đặc trưng. Chưa huấn luyện/chọn ngưỡng/đánh giá Test.
Đầu ra: D:\LUẬN VĂN\stock-news-anomaly-project\Stocks\notebooks\outputs\03_feature_engineering
